In [1]:
#!/usr/bin/env python3
"""
Simulate a PTA dataset with a single deterministic continuous-wave signal.

Requirements:
  enterprise
  enterprise_extensions
  libstempo
"""

import numpy as np
import libstempo as lt
from pathlib import Path
import pickle
from enterprise.pulsar import Pulsar
from enterprise_extensions.deterministic import cw_delay
import astropy.units as u
from astropy.coordinates import SkyCoord


In [2]:

# ---------------- CONFIG ---------------- #
BASE = Path("/scratch/na00078/projects/IPTA_MDC2/mdc2/group2/dataset_2")
PAR_DIR = BASE / "par"
TIM_DIR = BASE / "tim"
OUT_DIR = Path("/scratch/na00078/projects/IPTA_MDC2/IPTA_MDC2_data/sim_dataset2")
OUT_DIR.mkdir(exist_ok=True)
PKL_FILE = OUT_DIR / "G2D2_sim_injected_enterprise.pkl"

In [3]:

# CW parameters (dataset2)
CW = dict(
    fgw = 3.7e-9,                       # GW frequency [Hz]
    log10_h = -13.668773493298787,      # strain amplitude
    gw_phi = 3.3335788713091694,        # RA [rad]
    gw_theta = 0.6387905062299246,      # colatitude [rad]
    inclination = 0.8412486994612669,   # binary inclination [rad]
    phase0 = 0.24434609527920614,       # initial phase [rad]
    psi = 1.1187560505283651,           # polarization angle [rad]
)

# derived values
h0 = 10 ** CW["log10_h"]
gw_dec = np.pi/2 - CW["gw_theta"]
gw_ra  = CW["gw_phi"]

In [4]:
# ---------------- Helper: coordinates ---------------- #
def get_radec(psr):
    """Return pulsar RA, DEC [rad] whether par uses RAJ/DECJ or ELONG/ELAT."""
    pars = {p.upper(): p for p in psr.pars()}
    if "ELONG" in pars and "ELAT" in pars:
        elon = float(psr["ELONG"].val)
        elat = float(psr["ELAT"].val)
        c = SkyCoord(elon*u.rad, elat*u.rad, frame="barycentrictrueecliptic")
        return c.icrs.ra.rad, c.icrs.dec.rad
    if "RAJ" in pars and "DECJ" in pars:
        return float(psr["RAJ"].val), float(psr["DECJ"].val)
    if "RA" in pars and "DEC" in pars:
        return float(psr["RA"].val), float(psr["DEC"].val)
    raise KeyError(f"No RA/DEC or ELONG/ELAT for {psr.name}")

In [13]:
import numpy as np

def inject_cw(psr, cwpars):
    """Inject analytic Earth-term CW residuals (identical to Enterprise model)."""
    ra_p, dec_p = get_radec(psr)
    toas_sec = psr.stoas.copy().astype(float) * 86400.0
    t_ref = toas_sec.mean()
    t = toas_sec - t_ref

    # Unpack parameters
    h = 10**cwpars["log10_h"]
    fgw = cwpars["fgw"]
    omega = 2 * np.pi * fgw
    phase0 = cwpars["phase0"]
    psi = cwpars["psi"]
    iota = cwpars["inclination"]
    ra_gw = cwpars["gw_phi"]
    dec_gw = np.pi/2 - cwpars["gw_theta"]

    # Geometry
    k = np.array([
        np.cos(dec_gw) * np.cos(ra_gw),
        np.cos(dec_gw) * np.sin(ra_gw),
        np.sin(dec_gw)
    ])
    p = np.array([
        np.cos(dec_p) * np.cos(ra_p),
        np.cos(dec_p) * np.sin(ra_p),
        np.sin(dec_p)
    ])
    # Antenna pattern factors
    OmI = np.eye(3) - np.outer(k, k)
    e_plus = np.outer([np.sin(ra_gw), -np.cos(ra_gw), 0],
                      [-np.sin(dec_gw)*np.cos(ra_gw),
                       -np.sin(dec_gw)*np.sin(ra_gw),
                       np.cos(dec_gw)]) + 0
    e_cross = np.outer([np.sin(ra_gw), -np.cos(ra_gw), 0],
                       [np.cos(dec_gw)*np.cos(ra_gw),
                        np.cos(dec_gw)*np.sin(ra_gw),
                        np.sin(dec_gw)]) + 0

    # Rotate polarization by psi
    e_plus_rot = e_plus*np.cos(2*psi) - e_cross*np.sin(2*psi)
    e_cross_rot = e_plus*np.sin(2*psi) + e_cross*np.cos(2*psi)

    Fp = 0.5 * np.dot(p, np.dot(e_plus_rot, p)) / (1 + np.dot(k, p))
    Fx = 0.5 * np.dot(p, np.dot(e_cross_rot, p)) / (1 + np.dot(k, p))

    # CW waveform (Earth term only)
    A_plus = h * (1 + np.cos(iota)**2) / 2
    A_cross = -h * np.cos(iota)
    phase = omega * t + phase0
    res = (Fp * A_plus * np.sin(phase) + Fx * A_cross * np.cos(phase)) / omega

    psr.stoas[:] += res / 86400.0  # convert s→days
    return np.std(res) * 1e6


In [14]:
# ---------------- Build injected dataset ---------------- #
for par in sorted(PAR_DIR.glob("*.par")):
    name = par.stem
    tim = TIM_DIR / f"{name}.tim"
    psr = lt.tempopulsar(str(par), str(tim))
    rms = inject_cw(psr, CW)
    psr.savetim(str(OUT_DIR / f"{name}_enterprise_CW.tim"))
    print(f"{name}: injected RMS = {rms:.2f} µs")




J0030+0451: injected RMS = 0.03 µs
J0034-0534: injected RMS = 0.03 µs
J0218+4232: injected RMS = 0.05 µs
J0437-4715: injected RMS = 0.57 µs
J0613-0200: injected RMS = 0.03 µs
J0621+1002: injected RMS = 0.04 µs
J0711-6830: injected RMS = 0.30 µs
J0751+1807: injected RMS = 0.06 µs
J0900-3144: injected RMS = 0.13 µs
J1012+5307: injected RMS = 0.04 µs
J1022+1001: injected RMS = 0.06 µs
J1024-0719: injected RMS = 0.07 µs
J1045-4509: injected RMS = 0.09 µs
J1455-3330: injected RMS = 0.09 µs
J1600-3053: injected RMS = 0.12 µs
J1603-7202: injected RMS = 0.16 µs
J1614-2230: injected RMS = 0.10 µs
J1640+2224: injected RMS = 0.07 µs
J1643-1224: injected RMS = 0.08 µs
J1713+0747: injected RMS = 0.06 µs
J1730-2304: injected RMS = 0.11 µs
J1744-1134: injected RMS = 0.07 µs
J1857+0943: injected RMS = 0.04 µs
J1909-3744: injected RMS = 0.26 µs
J1918-0642: injected RMS = 0.05 µs
J1939+2134: injected RMS = 0.08 µs
J1944+0907: injected RMS = 0.06 µs
J2010-1323: injected RMS = 0.14 µs
J2124-3358: injected

In [15]:
# ---------------- Build Enterprise pickle ---------------- #
pulsars = []
for par in sorted(PAR_DIR.glob("*.par")):
    new_tim = OUT_DIR / f"{par.stem}_CW.tim"
    if new_tim.exists():
        psr_obj = Pulsar(str(par), str(new_tim), timing_package="tempo2")
        pulsars.append(psr_obj)

with open(PKL_FILE, "wb") as f:
    pickle.dump(pulsars, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"\nSaved {len(pulsars)} injected pulsars to {PKL_FILE}")


Saved 33 injected pulsars to /scratch/na00078/projects/IPTA_MDC2/IPTA_MDC2_data/sim_dataset2/G2D2_sim_injected_enterprise.pkl
